## 1️⃣ Setup Environment

In [ ]:
# Install ultralytics (YOLOv11)
!pip install -q ultralytics

# Import libraries
import os
import shutil
from pathlib import Path
from google.colab import drive
from ultralytics import YOLO
import torch

print("✅ Packages installed!")

In [ ]:
# Cek GPU
print("=" * 50)
print("INFO PERANGKAT")
print("=" * 50)

gpu_tersedia = torch.cuda.is_available()
print(f"GPU Tersedia: {gpu_tersedia}")

if gpu_tersedia:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    device = 0
else:
    print("⚠️ GPU tidak terdeteksi! Aktifkan GPU di Runtime → Change runtime type")
    device = "cpu"

print("=" * 50)

## 2️⃣ Mount Google Drive & Setup Dataset

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

In [ ]:
# ============================================
# KONFIGURASI PATH - SESUAIKAN JIKA PERLU!
# ============================================

# Path ke folder dataset di Google Drive
DRIVE_DATASET_PATH = "/content/drive/MyDrive/deteksi_kersen/train_split"

# Path lokal untuk training (lebih cepat)
LOCAL_DATASET_PATH = "/content/dataset"

# Path untuk menyimpan hasil
RESULTS_PATH = "/content/results"
DRIVE_RESULTS_PATH = "/content/drive/MyDrive/deteksi_kersen/results_colab"

print(f"📁 Dataset Drive: {DRIVE_DATASET_PATH}")
print(f"📁 Dataset Local: {LOCAL_DATASET_PATH}")
print(f"📁 Results: {RESULTS_PATH}")

In [ ]:
# Copy dataset dari Drive ke local storage (lebih cepat untuk training)
print("📂 Menyalin dataset ke local storage...")

if os.path.exists(LOCAL_DATASET_PATH):
    shutil.rmtree(LOCAL_DATASET_PATH)

shutil.copytree(DRIVE_DATASET_PATH, LOCAL_DATASET_PATH)
print("✅ Dataset berhasil disalin!")

# Verifikasi struktur dataset
print("\n📊 Struktur Dataset:")
!find {LOCAL_DATASET_PATH} -type d | head -20

In [ ]:
# Update data.yaml dengan path yang benar
data_yaml_content = f"""# Dataset Kersen untuk YOLOv11
path: {LOCAL_DATASET_PATH}
train: images/train
val: images/val
test: images/test

# Classes
nc: 3
names: ['mentah', 'setengah_matang', 'matang']
"""

data_yaml_path = f"{LOCAL_DATASET_PATH}/data.yaml"
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print("✅ data.yaml updated!")
print("\n📄 Isi data.yaml:")
print(data_yaml_content)

In [ ]:
# Verifikasi jumlah data
print("📊 Jumlah Data:")
print("=" * 40)

for split in ['train', 'val', 'test']:
    img_path = f"{LOCAL_DATASET_PATH}/images/{split}"
    label_path = f"{LOCAL_DATASET_PATH}/labels/{split}"

    n_images = len([f for f in os.listdir(img_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
    n_labels = len([f for f in os.listdir(label_path) if f.endswith('.txt')])

    status = "✅" if n_images == n_labels else "⚠️"
    print(f"{status} {split.upper():6}: {n_images} images, {n_labels} labels")

print("=" * 40)

## 3️⃣ Training YOLOv11

In [ ]:
# ============================================
# HAPUS HASIL TRAINING LAMA (FRESH START)
# ============================================

import shutil

# Path hasil training
train_result_path = f"{RESULTS_PATH}/kersen_yolo11"
drive_result_path = f"{DRIVE_RESULTS_PATH}/kersen_yolo11"

print("🗑️  Mengecek hasil training lama...")

# Hapus hasil lokal
if os.path.exists(train_result_path):
    shutil.rmtree(train_result_path)
    print(f"   ✅ Dihapus: {train_result_path}")
else:
    print(f"   ℹ️  Tidak ada: {train_result_path}")

# Hapus hasil di Google Drive
if os.path.exists(drive_result_path):
    shutil.rmtree(drive_result_path)
    print(f"   ✅ Dihapus: {drive_result_path}")
else:
    print(f"   ℹ️  Tidak ada: {drive_result_path}")

# Hapus folder runs jika ada (hasil training ultralytics)
runs_path = "/content/runs"
if os.path.exists(runs_path):
    shutil.rmtree(runs_path)
    print(f"   ✅ Dihapus: {runs_path}")

print("\n✅ Siap untuk training dari awal!")
print("=" * 50)

In [ ]:
# ============================================
# KONFIGURASI TRAINING
# ============================================

EPOCHS = 300             
BATCH_SIZE = 16           
IMG_SIZE = 640            
PATIENCE = 80             
LEARNING_RATE = 0.01      

print("=" * 50)
print("⚙️  KONFIGURASI TRAINING")
print("=" * 50)
print(f"Epochs: {EPOCHS} (Target akurasi 81%+)")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Image Size: {IMG_SIZE}")
print(f"Patience: {PATIENCE}")
print(f"Learning Rate: {LEARNING_RATE}")
print("=" * 50)
print()

# ============================================
# LOAD MODEL YOLO11m
# ============================================

from ultralytics import YOLO

print("📦 Loading YOLO11m Model (Medium - Balance antara akurasi & kecepatan)...")

# Download model YOLO11m
MODEL_NAME = "yolo11m.pt"
model = YOLO(MODEL_NAME)

print(f"✅ Model {MODEL_NAME} berhasil dimuat!")
print()

# Device
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Device: {device}")
if device == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
print("=" * 50)
print()

⚙️  KONFIGURASI TRAINING (OPTIMIZED)
Epochs: 200
Batch Size: 16
Image Size: 640
Patience: 50
Learning Rate: 0.01

📦 Loading YOLO11s Model (Small - lebih akurat)...
✅ Model yolo11s.pt berhasil dimuat!

🖥️  Device: cpu



## 🎯 Pilihan Model

### Pilih salah satu:

1. **YOLO11s** (Recommended) - Balance antara akurasi & kecepatan
   - Ukuran: ~22MB
   - Kecepatan: Fast
   - Akurasi: Tinggi (target 81%+)
   - **Gunakan cell di bawah ini**

2. **YOLO11m** (Akurasi Maksimal) - Jika YOLO11s belum cukup
   - Ukuran: ~50MB  
   - Kecepatan: Medium
   - Akurasi: Sangat Tinggi (target 85%+)
   - **Ubah MODEL_NAME ke "yolo11m.pt"** di cell selanjutnya

### Rekomendasi:
- Mulai dengan **YOLO11s** dulu
- Jika akurasi masih < 80%, coba **YOLO11m**

In [ ]:
# ============================================
# MULAI TRAINING
# ============================================

print("🚀 Memulai Training dengan Parameter Ultra Optimal...")
print("⏱️  Estimasi waktu: 60-120 menit dengan GPU T4 (300 epochs)")
print("=" * 50)

results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    device=device,
    project=RESULTS_PATH,
    name='kersen_yolo11',
    save=True,
    verbose=True,
    pretrained=True,
    optimizer='AdamW',       
    lr0=LEARNING_RATE,       
    lrf=0.0001,              
    momentum=0.937,          
    weight_decay=0.0005,     
    warmup_epochs=5.0,       
    warmup_momentum=0.8,     
    warmup_bias_lr=0.1,      
    box=7.5,                 
    cls=0.5,                 
    dfl=1.5,                 
    plots=True,
    cache=True,              
    workers=8,               
    
    # Augmentasi SUPER AGRESIF untuk model robust & akurasi tinggi
    hsv_h=0.025,             
    hsv_s=0.9,               
    hsv_v=0.6,               
    degrees=30.0,            
    translate=0.2,           
    scale=0.7,               
    shear=10.0,              
    perspective=0.0005,      
    flipud=0.5,              
    fliplr=0.5,              
    mosaic=1.0,              
    mixup=0.2,               
    copy_paste=0.15,         
    auto_augment='randaugment',  
    erasing=0.4,             
    crop_fraction=1.0,       
    
    # Advanced settings untuk akurasi maksimal
    close_mosaic=15,         
    label_smoothing=0.05,    
    nbs=64,                  
    overlap_mask=True,       
    mask_ratio=4,            
    dropout=0.1,             
    val=True,                
    save_period=20,          
)

print("\n" + "=" * 50)
print("✅ Training Selesai!")
print("=" * 50)

In [ ]:
# ============================================
# BACKUP HASIL KE GOOGLE DRIVE
# ============================================

print("💾 Menyimpan hasil ke Google Drive...")

# Buat folder jika belum ada
os.makedirs(DRIVE_RESULTS_PATH, exist_ok=True)

# Copy hasil training ke Drive
train_result_path = f"{RESULTS_PATH}/kersen_yolo11"
drive_backup_path = f"{DRIVE_RESULTS_PATH}/kersen_yolo11"

if os.path.exists(train_result_path):
    if os.path.exists(drive_backup_path):
        shutil.rmtree(drive_backup_path)
    shutil.copytree(train_result_path, drive_backup_path)
    print(f"✅ Hasil disimpan ke: {drive_backup_path}")
else:
    print("⚠️ Folder hasil tidak ditemukan")

print("=" * 50)

## 4️⃣ Evaluasi Model

In [ ]:
# Load best model dan evaluasi
best_model_path = f"{RESULTS_PATH}/kersen_yolo11/weights/best.pt"

print("📊 Evaluasi Model pada Test Set...")
best_model = YOLO(best_model_path)

metrics = best_model.val(
    data=data_yaml_path,
    device=device,
    split='test'
)

print("\n" + "=" * 50)
print("📈 HASIL EVALUASI")
print("=" * 50)
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print("=" * 50)

In [ ]:
# ============================================
# EVALUASI MENDALAM & ANALISIS CONFIDENCE
# ============================================

print("📊 Evaluasi Model pada Test Set dengan Berbagai Confidence...")
print("=" * 50)

best_model = YOLO(best_model_path)

# Test dengan berbagai confidence threshold
confidence_levels = [0.50, 0.60, 0.70, 0.75, 0.80]
results_summary = []

for conf in confidence_levels:
    metrics = best_model.val(
        data=data_yaml_path,
        device=device,
        split='test',
        conf=conf,
        iou=0.4,
        verbose=False
    )
    
    results_summary.append({
        'confidence': conf,
        'mAP50': metrics.box.map50,
        'mAP50-95': metrics.box.map,
        'precision': metrics.box.mp,
        'recall': metrics.box.mr
    })
    
    print(f"Conf {conf:.2f} | mAP50: {metrics.box.map50:.4f} | Precision: {metrics.box.mp:.4f} | Recall: {metrics.box.mr:.4f}")

print("\n" + "=" * 50)
print("🎯 REKOMENDASI CONFIDENCE THRESHOLD:")
print("=" * 50)

# Cari confidence optimal (balance antara precision & recall)
best_conf = max(results_summary, key=lambda x: (x['precision'] + x['recall']) / 2)
print(f"✅ Confidence Optimal: {best_conf['confidence']:.2f}")
print(f"   - mAP50: {best_conf['mAP50']:.4f}")
print(f"   - Precision: {best_conf['precision']:.4f}")
print(f"   - Recall: {best_conf['recall']:.4f}")
print("\n💡 Gunakan confidence {:.2f} di aplikasi Flask untuk hasil terbaik!".format(best_conf['confidence']))
print("=" * 50)

In [ ]:
# Tampilkan grafik training
from IPython.display import Image, display

print("📊 Grafik Training:")

# Results plot
results_img = f"{RESULTS_PATH}/kersen_yolo11/results.png"
if os.path.exists(results_img):
    display(Image(filename=results_img, width=800))

# Confusion matrix
cm_img = f"{RESULTS_PATH}/kersen_yolo11/confusion_matrix.png"
if os.path.exists(cm_img):
    print("\n📊 Confusion Matrix:")
    display(Image(filename=cm_img, width=600))

## 5️⃣ Simpan & Download Model

In [ ]:
# Download model best.pt langsung
from google.colab import files

print("📥 Download model...")

# Rename dan download
final_model_path = f"{RESULTS_PATH}/yolo11s_kersen_best.pt"
shutil.copy(best_model_path, final_model_path)

files.download(final_model_path)
print("✅ Model downloaded: yolo11s_kersen_best.pt")

In [ ]:
# ============================================
# EXPORT MODEL & KONFIGURASI OPTIMAL
# ============================================

from datetime import datetime

print("📋 Menyimpan informasi konfigurasi optimal...")

# Buat file konfigurasi untuk aplikasi
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

config_info = f"""
# ==================================================
# KONFIGURASI OPTIMAL HASIL TRAINING
# ==================================================
# Generated: {current_time}

Model: yolo11s_kersen_best.pt

METRICS (Test Set):
- mAP50: {metrics.box.map50:.4f}
- mAP50-95: {metrics.box.map:.4f}
- Precision: {metrics.box.mp:.4f}
- Recall: {metrics.box.mr:.4f}

REKOMENDASI UNTUK APLIKASI:
- Confidence Threshold: {best_conf['confidence']:.2f}
- IOU Threshold: 0.40
- Preprocessing: 
  * Contrast enhancement: alpha=1.2, beta=15
  * Denoising: fastNlMeansDenoisingColored
  * Saturation boost: 1.3x
  
TRAINING PARAMETERS:
- Epochs: {EPOCHS}
- Batch Size: {BATCH_SIZE}
- Image Size: {IMG_SIZE}
- Optimizer: AdamW
- Learning Rate: {LEARNING_RATE}
- Augmentation: Super Aggressive

TARGET AKURASI:
- Minimum Confidence: 70%
- Average Confidence: 81%+

CARA PAKAI:
1. Download yolo11s_kersen_best.pt
2. Letakkan di folder models/
3. Update app.py dengan confidence={best_conf['confidence']:.2f}
4. Restart Flask app

# ==================================================
"""

config_path = f"{RESULTS_PATH}/CONFIG_OPTIMAL.txt"
with open(config_path, 'w') as f:
    f.write(config_info)

print(config_info)

# Copy ke Drive juga
shutil.copy(config_path, f"{DRIVE_RESULTS_PATH}/CONFIG_OPTIMAL.txt")
print(f"✅ Konfigurasi disimpan ke: {config_path}")
print(f"✅ Backup ke Drive: {DRIVE_RESULTS_PATH}/CONFIG_OPTIMAL.txt")

## 6️⃣ Test Inference (Opsional)

In [ ]:
# Test inference pada beberapa gambar dengan confidence threshold tinggi
import glob

print("🎯 Test Inference dengan Confidence Threshold Tinggi...")

# Ambil beberapa gambar test
test_images = glob.glob(f"{LOCAL_DATASET_PATH}/images/test/*.jpg")[:5]

if test_images:
    # Test dengan berbagai confidence threshold
    for conf in [0.70, 0.75, 0.80]:
        print(f"\n📊 Testing dengan confidence = {conf}")
        results = best_model.predict(
            source=test_images,
            conf=conf,
            iou=0.4,           # IOU threshold untuk NMS
            save=True,
            project=RESULTS_PATH,
            name=f'test_inference_conf{int(conf*100)}',
            augment=True,      # TTA untuk inference lebih akurat
        )
        
        print(f"✅ Hasil disimpan di: {RESULTS_PATH}/test_inference_conf{int(conf*100)}/")

    # Tampilkan hasil dengan confidence 0.75 (target)
    inference_dir = f"{RESULTS_PATH}/test_inference_conf75/"
    if os.path.exists(inference_dir):
        print(f"\n📷 Hasil Inference (Confidence 0.75):")
        for img_path in glob.glob(f"{inference_dir}/*.jpg")[:3]:
            print(f"\n🖼️  {os.path.basename(img_path)}")
            display(Image(filename=img_path, width=500))
else:
    print("⚠️ Tidak ada gambar test ditemukan")

---

## ✅ Selesai!

### 📋 Langkah Selanjutnya:

1. **Download** file `yolo11s_kersen_best.pt`
2. **Pindahkan** ke folder `models/` di project lokal
3. **Rename** menjadi `yolo11s_kersen_best.pt` (jika berbeda)
4. **Jalankan** `app.py` untuk testing!

### 📁 Struktur File di Google Drive:
```
MyDrive/deteksi_kersen/
├── train_split/          # Dataset
└── results_colab/        # Hasil training
    └── kersen_yolo11/
        ├── weights/
        │   ├── best.pt   # Model terbaik ⭐
        │   └── last.pt
        ├── results.png
        ├── confusion_matrix.png
        └── ...
```

---

## 💡 Peningkatan yang Diterapkan untuk Akurasi 81%+

### ✅ Optimasi Training (Sudah Diterapkan):

#### 1. **Epochs & Patience**
   - Epochs: 200 → **300** (learning lebih dalam)
   - Patience: 50 → **80** (konvergensi sempurna)
   - Close mosaic: 15 epoch terakhir untuk fine-tuning

#### 2. **Augmentasi SUPER Agresif**
   - HSV Saturation: **0.9** (sangat penting untuk deteksi kematangan berdasarkan warna!)
   - Rotation: **30°**
   - Mosaic: **1.0** (sangat efektif)
   - Mixup: **0.2**
   - Copy-paste: **0.15**
   - Random Augment Policy: **randaugment**
   - Random Erasing: **0.4**

#### 3. **Loss Functions Optimization**
   - Box loss gain: **7.5** (fokus pada akurasi posisi)
   - DFL gain: **1.5** (distribution focal loss)
   - Label smoothing: **0.05** (generalisasi lebih baik)

#### 4. **Regularization**
   - Dropout: **0.1** (mencegah overfitting)
   - Weight decay: **0.0005**

#### 5. **Optimizer Settings**
   - Optimizer: **AdamW** (terbaik untuk akurasi tinggi)
   - Learning rate decay: 0.01 → 0.0001 (smooth decay)
   - Warmup: **5 epochs**

### 🎯 Target Hasil Training:

| Metric | Target | Keterangan |
|--------|--------|------------|
| **mAP50** | ≥ 0.88 | Akurasi deteksi pada IOU 0.5 |
| **mAP50-95** | ≥ 0.72 | Rata-rata mAP (standar COCO) |
| **Precision** | ≥ 0.85 | Presisi minimal 85% |
| **Recall** | ≥ 0.83 | Recall minimal 83% |
| **Inference Confidence** | 70-80% | Confidence saat deteksi real-time |

### 📊 Ekspektasi:
- Training dengan konfigurasi ini akan menghasilkan model yang **konsisten mendeteksi dengan confidence 75-85%**
- False positive akan **minimal** karena augmentasi yang kuat
- Model akan **robust** terhadap variasi pencahayaan dan sudut kamera

### 🚀 Setelah Training:
1. Download file `yolo11s_kersen_best.pt`
2. Pindahkan ke folder `models/` di project lokal
3. Restart aplikasi Flask
4. Gunakan confidence threshold 0.75 di UI
5. Akurasi deteksi seharusnya **rata-rata 81%+** dengan minimum **70%**

### 💪 Tips Tambahan:
- Pastikan dataset memiliki **minimal 200-300 gambar per kelas**
- Gunakan pencahayaan yang **konsisten dan baik** saat foto
- Anotasi harus **presisi** (bounding box pas dengan objek)
- Jika masih kurang, pertimbangkan **YOLO11m** (model lebih besar, lebih akurat)